# 01 - Exploratory Data Analysis
Run from the `notebooks/` folder. Describe results using what YOUR run shows.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

path = Path('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(path)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()
print('Duplicate rows:', df.duplicated().sum())
print('customerID unique:', df['customerID'].is_unique)
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# TotalCharges is text with blanks in the raw file
blank = (df['TotalCharges'].str.strip() == '').sum()
print('Blank TotalCharges:', blank)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df[df['TotalCharges'].isna()][['tenure', 'MonthlyCharges', 'Churn']]

In [ ]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(3))
sns.countplot(data=df, x='Churn'); plt.title('Churn class distribution'); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x='Churn', y='tenure', ax=ax[0]); ax[0].set_title('Tenure by churn')
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=ax[1]); ax[1].set_title('Monthly charges by churn')
plt.tight_layout(); plt.savefig('../reports/figures/tenure_charges_by_churn.png', dpi=150); plt.show()

In [ ]:
for col in ['Contract', 'InternetService', 'PaymentMethod']:
    rate = (df['Churn'].eq('Yes').groupby(df[col]).mean() * 100).round(1)
    print(f'\nChurn rate (%) by {col}:\n', rate.sort_values(ascending=False))
    rate.sort_values().plot(kind='barh', title=f'Churn rate (%) by {col}')
    plt.tight_layout(); plt.savefig(f'../reports/figures/churn_by_{col}.png', dpi=150); plt.show()

In [ ]:
df['tenure_band'] = pd.cut(df['tenure'], [-1, 6, 12, 24, 48, 72], labels=['0-6', '7-12', '13-24', '25-48', '49-72'])
(df['Churn'].eq('Yes').groupby(df['tenure_band'], observed=True).mean() * 100).round(1).plot(kind='bar', title='Churn rate (%) by tenure band')
plt.tight_layout(); plt.show()

In [ ]:
sns.heatmap(df[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']].corr(), annot=True, fmt='.2f', cmap='Blues')
plt.title('Numeric correlations'); plt.show()

- The dataset has ___ customers; ___% churned, so the classes are imbalanced.
- ___ rows have blank TotalCharges. They are all customers with tenure 0 (new customers).
- Churn rate by contract: month-to-month ___%, one year ___%, two year ___%.
- Churn rate by internet service: fiber optic ___%, DSL ___%, no internet ___%.
- Churn is highest in the ___ tenure band and lowest in the ___ band.
- These are patterns in historical data; they don't prove what causes churn.